In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

print(metadata_df.shape)
metadata_df.head()


(47, 9)


,sample_title,tissue,age at_diag,age at_first_event_or_censor,ethnicity,efstime,stime,efscens,scens
samples,,,,,,,,,
GSM6751285,Neuroblastoma_usi_PALUKI,Neuroblastoma,1110,5152,Not Hispanic or Latino,4043,4043,0,0
GSM6751286,Neuroblastoma_usi_PALWKV,Neuroblastoma,2019,2263,Not Hispanic or Latino,245,245,0,0
GSM6751287,Neuroblastoma_usi_PAMKGY,Neuroblastoma,926,1131,Not Hispanic or Latino,206,206,1,1
GSM6751288,Neuroblastoma_usi_PAMMGU,Neuroblastoma,953,3418,Hispanic or Latino,2466,2466,0,0
GSM6751289,Neuroblastoma_usi_PANLCE,Neuroblastoma,1960,5797,Not Hispanic or Latino,3838,3838,0,0


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

print(df_combined.shape)
df_combined.head()

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_6251/3782264241.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


(47, 39385)


,sample_title,tissue,age at_diag,age at_first_event_or_censor,ethnicity,efstime,stime,efscens,scens,DDX11L1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM6751285,Neuroblastoma_usi_PALUKI,Neuroblastoma,1110,5152,Not Hispanic or Latino,4043,4043,0,0,9,...,28001,94,116,122,12986,2182,642,17253,210,2309
GSM6751286,Neuroblastoma_usi_PALWKV,Neuroblastoma,2019,2263,Not Hispanic or Latino,245,245,0,0,5,...,17503,23,25,35,7566,1713,269,8324,60,610
GSM6751287,Neuroblastoma_usi_PAMKGY,Neuroblastoma,926,1131,Not Hispanic or Latino,206,206,1,1,10,...,5945,20,22,20,3335,590,206,3363,66,720
GSM6751288,Neuroblastoma_usi_PAMMGU,Neuroblastoma,953,3418,Hispanic or Latino,2466,2466,0,0,5,...,12681,32,42,46,6635,1011,289,8168,62,902
GSM6751289,Neuroblastoma_usi_PANLCE,Neuroblastoma,1960,5797,Not Hispanic or Latino,3838,3838,0,0,5,...,13932,41,57,71,7582,1255,413,5627,104,1014


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
